# Notebook 8 — Panoptic Quality Evaluation

**Stage GT file · runs last (after Notebook 7)**

> **Pipeline execution order: `6 -> 1 -> 2 -> 4 -> 7 -> 8`.**
> Notebook 8 scores each segmentation method's room-label map against the Notebook 7 ground
> truth with Panoptic Quality (SQ / RQ / PQ).

## Purpose
Compute SQ, RQ, PQ for each segmentation method vs. the GT room labels.

## Inputs
- `stage_gt/gt_room_labels.npy` (Notebook 7).
- `stage2_watershed/room_labels.npy` (**Geometry**).
- `stage4_sam_refined/room_labels_refined.npy` (**SAM** / **Geometry + SAM**).

## Outputs  (`{out_root}/stage_gt/`)
- `pq_results.json` — per-method `dict(SQ, RQ, PQ, TP, FP, FN)`.

## Assumptions
- All label maps share the Stage 1 grid (same `H x W`, same transform), so pixels correspond.
- Matching threshold is `IoU > 0.5` (standard panoptic). "SAM" and "Geometry + SAM" both read
  the Notebook 4 refined map here — Notebook 4 refines the watershed with SAM, so that single
  map **is** the hybrid. Cleanly separating "purely SAM" from "Geometry + SAM" needs two
  different pipeline configurations; each row is labelled so the source is unambiguous.

### Setup

In [1]:
# ============================== scan2bim setup (local) ==============================
import os
import json
import numpy as np
import scan2bim
from scan2bim import artifacts as A

CFG = scan2bim.load_config()
SHOW_DEBUG = True
STAGE_GT = 'stage_gt'
print('scan2bim', scan2bim.__version__, '| output root:', CFG.out_root)

scan2bim 1.0.0 | output root: C:\onestruction\scan2bim_out


### Step 1 — Load the GT and the three prediction maps

In [2]:
gt_dir = A.stage_dir(CFG.out_root, STAGE_GT)
gt_labels = A.load_npy(os.path.join(gt_dir, 'gt_room_labels.npy')).astype('int32')

s2 = A.load_stage_dir(CFG.out_root, A.STAGE2)
s4 = A.load_stage_dir(CFG.out_root, A.STAGE4)
geom = A.load_npy(os.path.join(s2, A.ROOM_LABELS_NPY)).astype('int32')
sam = A.load_npy(os.path.join(s4, A.REFINED_LABELS_NPY)).astype('int32')

# NB4 refines the watershed with SAM, so the refined map is BOTH the 'SAM' and the
# 'Geometry + SAM' result; truly separating them needs two pipeline configs (see header).
methods = {
    'Geometry':       geom,
    'SAM':            sam,
    'Geometry + SAM': sam,
}
for name, lab in methods.items():
    assert lab.shape == gt_labels.shape, (name, lab.shape, gt_labels.shape)
    print('%-16s rooms=%3d  shape=%s'
          % (name, len([r for r in np.unique(lab) if r >= 1]), lab.shape))
print('GT rooms          :', len([r for r in np.unique(gt_labels) if r >= 1]))

Geometry         rooms= 32  shape=(856, 1011)
SAM              rooms= 32  shape=(856, 1011)
Geometry + SAM   rooms= 32  shape=(856, 1011)
GT rooms          : 23


### Step 2 — `compute_pq` (full IoU matrix + greedy matching at `IoU > 0.5`)
For every `(pred_room, gt_room)` pair, `IoU = |pred ∩ gt| / |pred ∪ gt|`. Pairs are sorted by
IoU descending and assigned greedily; a pair with `IoU > 0.5` is a true positive (each GT and
each pred matched at most once). `SQ = mean IoU over TPs`, `RQ = TP / (TP + 0.5·FP + 0.5·FN)`,
`PQ = SQ · RQ`.

In [3]:
def compute_pq(pred_labels, gt_labels):
    pred_ids = np.array([i for i in np.unique(pred_labels) if i >= 1])
    gt_ids = np.array([i for i in np.unique(gt_labels) if i >= 1])
    P, G = len(pred_ids), len(gt_ids)
    if P == 0 or G == 0:
        return dict(SQ=0.0, RQ=0.0, PQ=0.0, TP=0, FP=int(P), FN=int(G))

    pred_area = np.array([(pred_labels == v).sum() for v in pred_ids], np.int64)
    gt_area = np.array([(gt_labels == v).sum() for v in gt_ids], np.int64)

    inter = np.zeros((P, G), np.int64)
    both = (pred_labels >= 1) & (gt_labels >= 1)
    if both.any():
        # pred_ids / gt_ids are sorted (np.unique), so searchsorted maps a label to its index.
        pr = np.searchsorted(pred_ids, pred_labels[both])
        gr = np.searchsorted(gt_ids, gt_labels[both])
        np.add.at(inter, (pr, gr), 1)

    union = pred_area[:, None] + gt_area[None, :] - inter
    iou = np.where(union > 0, inter / union, 0.0)

    pairs = sorted(((iou[i, j], i, j)
                    for i in range(P) for j in range(G) if iou[i, j] > 0.5),
                   reverse=True)
    pred_taken, gt_taken, matched = set(), set(), []
    for v, i, j in pairs:
        if i in pred_taken or j in gt_taken:
            continue
        pred_taken.add(i); gt_taken.add(j); matched.append(v)

    TP = len(matched); FP = P - TP; FN = G - TP
    SQ = float(np.mean(matched)) if TP else 0.0
    denom = TP + 0.5 * FP + 0.5 * FN
    RQ = TP / denom if denom else 0.0
    return dict(SQ=SQ, RQ=RQ, PQ=SQ * RQ, TP=int(TP), FP=int(FP), FN=int(FN))

### Step 3 — Run `compute_pq` for each method + print the results table

In [4]:
results = {name: compute_pq(lab, gt_labels) for name, lab in methods.items()}

print('Method              SQ      RQ      PQ      TP   FP   FN')
print('-' * 58)
for name, r in results.items():
    print('%-16s  %5.1f%%  %5.1f%%  %5.1f%%   %3d  %3d  %3d'
          % (name, 100 * r['SQ'], 100 * r['RQ'], 100 * r['PQ'],
             r['TP'], r['FP'], r['FN']))

Method              SQ      RQ      PQ      TP   FP   FN


----------------------------------------------------------
Geometry           73.8%   54.5%   40.3%    15   17    8
SAM                73.8%   54.5%   40.3%    15   17    8
Geometry + SAM     73.8%   54.5%   40.3%    15   17    8


### Step 4 — Save `pq_results.json`

In [5]:
out_dir = A.ensure_dir(gt_dir)
res_path = os.path.join(out_dir, 'pq_results.json')
with open(res_path, 'w') as f:
    json.dump(results, f, indent=2)
print('packaged ->', res_path)

packaged -> C:\onestruction\scan2bim_out\stage_gt\pq_results.json
